# Tải 4 Dataset (idempotent)

Notebook này tải 4 dataset cho đề tài phát hiện ảnh khuôn mặt AI.

- Nếu dataset đã tồn tại trong `data/raw/<folder>` thì sẽ **bỏ qua**, không tải lại.
- Điền Kaggle credentials vào cell bên dưới — token được ghi tạm vào `~/.kaggle/kaggle.json`.
- Chạy cell cleanup ở cuối để xoá token sau khi tải xong.

## ⚠️ Điền credentials vào đây trước khi chạy các cell còn lại

In [1]:
# Điền Kaggle username và key vào đây
# Lấy key tại: https://www.kaggle.com/settings → Account → API → Create New Token
KAGGLE_USERNAME = "trongnguyenminh07"   # ← điền vào
KAGGLE_KEY      = "KGAT_0f65ac419a7242432d2618b8fb810a2c"   # ← điền vào

In [2]:
%pip install -q kaggle

from pathlib import Path
import json
import os
import subprocess

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_RAW.mkdir(parents=True, exist_ok=True)

KAGGLE_FILE = Path.home() / '.kaggle' / 'kaggle.json'
_KAGGLE_CREATED_BY_NOTEBOOK = False

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATA_RAW     =', DATA_RAW)

Note: you may need to restart the kernel to use updated packages.
PROJECT_ROOT = D:\UIT\XyLyDuLieuLon_Thu5\DS200.F21.CN2.BigData
DATA_RAW     = D:\UIT\XyLyDuLieuLon_Thu5\DS200.F21.CN2.BigData\data\raw



[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
DATASETS = [
    {
        'name': '140k Real and Fake Faces',
        'folder': '140k-real-and-fake-faces',
        'slug': 'xhlulu/140k-real-and-fake-faces'
    },
    {
        'name': 'Deepfake and Real Images',
        'folder': 'deepfake-and-real-images',
        'slug': 'manjilkarki/deepfake-and-real-images'
    },
    {
        'name': 'Fake-Vs-Real-Faces (Hard)',
        'folder': 'hardfakevsrealfaces',
        'slug': 'hamzaboulahia/hardfakevsrealfaces'
    },
    {
        'name': 'Real and Fake Face Detection',
        'folder': 'real-and-fake-face-detection',
        'slug': 'ciplab/real-and-fake-face-detection'
    },
]

def has_existing_content(folder: Path) -> bool:
    return folder.exists() and any(p.is_file() for p in folder.rglob('*'))

def setup_kaggle_credentials():
    global _KAGGLE_CREATED_BY_NOTEBOOK
    if KAGGLE_FILE.exists():
        print(f'Dùng token có sẵn: {KAGGLE_FILE}')
        return
    if not KAGGLE_USERNAME or not KAGGLE_KEY:
        raise ValueError('Chưa điền KAGGLE_USERNAME / KAGGLE_KEY ở cell đầu.')
    KAGGLE_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(KAGGLE_FILE, 'w', encoding='utf-8') as f:
        json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
    os.chmod(KAGGLE_FILE, 0o600)
    _KAGGLE_CREATED_BY_NOTEBOOK = True
    print(f'Đã tạo token tạm: {KAGGLE_FILE}')

def cleanup_kaggle_credentials(force=False):
    if KAGGLE_FILE.exists() and (_KAGGLE_CREATED_BY_NOTEBOOK or force):
        KAGGLE_FILE.unlink()
        print(f'Đã xoá token: {KAGGLE_FILE}')
    else:
        print('Không có token tạm để xoá.')

def download_kaggle(slug: str, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = ['kaggle', 'datasets', 'download', '-d', slug, '-p', str(out_dir), '--unzip']
    print('RUN:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

In [4]:
pending = [d for d in DATASETS if not has_existing_content(DATA_RAW / d['folder'])]

if pending:
    setup_kaggle_credentials()

for ds in DATASETS:
    target_dir = DATA_RAW / ds['folder']
    if has_existing_content(target_dir):
        print(f"[SKIP] {ds['name']} — đã có dữ liệu")
        continue
    print(f"[DOWN] {ds['name']}")
    download_kaggle(ds['slug'], target_dir)

print('\nHoàn tất.')

Đã tạo token tạm: C:\Users\Admin\.kaggle\kaggle.json
[DOWN] 140k Real and Fake Faces
RUN: kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p D:\UIT\XyLyDuLieuLon_Thu5\DS200.F21.CN2.BigData\data\raw\140k-real-and-fake-faces --unzip
[DOWN] Deepfake and Real Images
RUN: kaggle datasets download -d manjilkarki/deepfake-and-real-images -p D:\UIT\XyLyDuLieuLon_Thu5\DS200.F21.CN2.BigData\data\raw\deepfake-and-real-images --unzip
[DOWN] Fake-Vs-Real-Faces (Hard)
RUN: kaggle datasets download -d hamzaboulahia/hardfakevsrealfaces -p D:\UIT\XyLyDuLieuLon_Thu5\DS200.F21.CN2.BigData\data\raw\hardfakevsrealfaces --unzip
[DOWN] Real and Fake Face Detection
RUN: kaggle datasets download -d ciplab/real-and-fake-face-detection -p D:\UIT\XyLyDuLieuLon_Thu5\DS200.F21.CN2.BigData\data\raw\real-and-fake-face-detection --unzip

Hoàn tất.


In [5]:
# Chay cell nay neu ban muon xoa token ngay sau khi download xong.
cleanup_kaggle_credentials(force=False)

Đã xoá token: C:\Users\Admin\.kaggle\kaggle.json
